<a href="https://colab.research.google.com/github/crammiee/170melanoma/blob/main/170melanoma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

FILE_FEATURES = '/content/drive/My Drive/Melanoma170/train-metadata.csv'
FILE_LABELS   = '/content/drive/My Drive/Melanoma170/train-labels.csv'

df_features = pd.read_csv(FILE_FEATURES, nrows=5)
df_labels   = pd.read_csv(FILE_LABELS, nrows=5)

print("=== FEATURES COLUMNS ===")
print(df_features.columns.tolist())
print("\n=== LABELS COLUMNS ===")
print(df_labels.columns.tolist())

print("\n=== FEATURES SHAPE ===")
df_features_full = pd.read_csv(FILE_FEATURES)
print(df_features_full.shape)

print("\n=== LABELS SHAPE ===")
df_labels_full = pd.read_csv(FILE_LABELS)
print(df_labels_full.shape)

Mounted at /content/drive
=== FEATURES COLUMNS ===
['isic_id', 'patient_id', 'age_approx', 'sex', 'anatom_site_general', 'clin_size_long_diam_mm', 'image_type', 'tbp_tile_type', 'tbp_lv_A', 'tbp_lv_Aext', 'tbp_lv_B', 'tbp_lv_Bext', 'tbp_lv_C', 'tbp_lv_Cext', 'tbp_lv_H', 'tbp_lv_Hext', 'tbp_lv_L', 'tbp_lv_Lext', 'tbp_lv_areaMM2', 'tbp_lv_area_perim_ratio', 'tbp_lv_color_std_mean', 'tbp_lv_deltaA', 'tbp_lv_deltaB', 'tbp_lv_deltaL', 'tbp_lv_deltaLB', 'tbp_lv_deltaLBnorm', 'tbp_lv_eccentricity', 'tbp_lv_location', 'tbp_lv_location_simple', 'tbp_lv_minorAxisMM', 'tbp_lv_nevi_confidence', 'tbp_lv_norm_border', 'tbp_lv_norm_color', 'tbp_lv_perimeterMM', 'tbp_lv_radial_color_std_max', 'tbp_lv_stdL', 'tbp_lv_stdLExt', 'tbp_lv_symm_2axis', 'tbp_lv_symm_2axis_angle', 'tbp_lv_x', 'tbp_lv_y', 'tbp_lv_z']

=== LABELS COLUMNS ===
['isic_id', 'malignant']

=== FEATURES SHAPE ===
(401059, 42)

=== LABELS SHAPE ===
(401059, 2)


# Melanoma Risk Analysis
## Step 1 — Merge Both CSV Files

We load the features and labels files separately, then join them using `isic_id` as the key.
After merging, we check how many rows were retained to make sure nothing was lost.

In [2]:
df = pd.merge(df_features_full, df_labels_full, on='isic_id', how='inner')
print(f"Merged shape: {df.shape}")
print(f"Rows lost: {len(df_features_full) - len(df)}")

Merged shape: (401059, 43)
Rows lost: 0


## Step 2 — Drop Non-Feature Columns

We remove columns that carry no analytical value:
- `isic_id` — just an identifier, no longer needed after the merge
- `image_type` and `tbp_tile_type` — uniform across all records, zero signal
- `patient_id` — saved separately for later use in model cross-validation

In [3]:
patient_id = df.pop('patient_id')
df = df.drop(columns=['isic_id', 'image_type', 'tbp_tile_type'])
print(f"Remaining columns: {df.shape[1]}")
print(df.columns.tolist())

Remaining columns: 39
['age_approx', 'sex', 'anatom_site_general', 'clin_size_long_diam_mm', 'tbp_lv_A', 'tbp_lv_Aext', 'tbp_lv_B', 'tbp_lv_Bext', 'tbp_lv_C', 'tbp_lv_Cext', 'tbp_lv_H', 'tbp_lv_Hext', 'tbp_lv_L', 'tbp_lv_Lext', 'tbp_lv_areaMM2', 'tbp_lv_area_perim_ratio', 'tbp_lv_color_std_mean', 'tbp_lv_deltaA', 'tbp_lv_deltaB', 'tbp_lv_deltaL', 'tbp_lv_deltaLB', 'tbp_lv_deltaLBnorm', 'tbp_lv_eccentricity', 'tbp_lv_location', 'tbp_lv_location_simple', 'tbp_lv_minorAxisMM', 'tbp_lv_nevi_confidence', 'tbp_lv_norm_border', 'tbp_lv_norm_color', 'tbp_lv_perimeterMM', 'tbp_lv_radial_color_std_max', 'tbp_lv_stdL', 'tbp_lv_stdLExt', 'tbp_lv_symm_2axis', 'tbp_lv_symm_2axis_angle', 'tbp_lv_x', 'tbp_lv_y', 'tbp_lv_z', 'malignant']


## Step 3 — Separate Target from Features

We isolate the label column `malignant` as our target variable `y`.
Everything else becomes our feature matrix `X`.

In [4]:
y = df['malignant']
X = df.drop(columns=['malignant'])
print(f"Features shape: {X.shape}")
print(f"Malignant count: {y.sum()}")
print(f"Benign count: {(y == 0).sum()}")

Features shape: (401059, 38)
Malignant count: 393.0
Benign count: 400666


## Step 4 — Handle Missing Values

Missing values are filled before any splitting or analysis:
- Numeric columns → filled with the **median** (robust to outliers)
- Categorical columns → filled with the **mode** (most frequent value)

In [5]:
numeric_cols = X.select_dtypes(include='number').columns
categorical_cols = X.select_dtypes(include='object').columns

X[numeric_cols] = X[numeric_cols].fillna(X[numeric_cols].median())
X[categorical_cols] = X[categorical_cols].fillna(X[categorical_cols].mode().iloc[0])

print("Missing values after imputation:")
print(X.isnull().sum().sum())

Missing values after imputation:
0


## Step 5 — Sanity Check: Clip Out-of-Range Values

Some numeric columns have biologically impossible values due to recording errors.
We clip them to their valid ranges:
- `age_approx` → 0 to 85
- `tbp_lv_nevi_confidence` → 0 to 100
- `tbp_lv_eccentricity` → 0 to 1
- `tbp_lv_symm_2axis` → 0 to 1

In [6]:
X['age_approx'] = X['age_approx'].clip(0, 85)
X['tbp_lv_nevi_confidence'] = X['tbp_lv_nevi_confidence'].clip(0, 100)
X['tbp_lv_eccentricity'] = X['tbp_lv_eccentricity'].clip(0, 1)
X['tbp_lv_symm_2axis'] = X['tbp_lv_symm_2axis'].clip(0, 1)

print("Clipping done. Sample check:")
print(X[['age_approx','tbp_lv_nevi_confidence','tbp_lv_eccentricity','tbp_lv_symm_2axis']].describe())

Clipping done. Sample check:
          age_approx  tbp_lv_nevi_confidence  tbp_lv_eccentricity  \
count  401059.000000           401059.000000        401059.000000   
mean       58.026849               38.520265             0.741238   
std        13.549664               41.480936             0.143857   
min         5.000000                0.000000             0.027667   
25%        50.000000                0.109819             0.656627   
50%        60.000000               14.408514             0.768215   
75%        70.000000               87.791395             0.853175   
max        85.000000              100.000000             0.974960   

       tbp_lv_symm_2axis  
count      401059.000000  
mean            0.306823  
std             0.125038  
min             0.052034  
25%             0.211429  
50%             0.282297  
75%             0.382022  
max             0.977055  


## Step 6 — Feature Engineering

We create new columns derived from existing ones to better capture lesion characteristics:
- `color_contrast_3d` — 3D color distance between lesion and surrounding skin
- `elongation` — how stretched the lesion is (minor axis vs long diameter)
- `nevi_color_tension` — interaction between nevus confidence and color irregularity
- `log_area` — square root of area to reduce skew
- `compactness` — how circular vs irregular the lesion shape is
- `chroma_contrast` — difference in color saturation between lesion interior and exterior

In [7]:
X['color_contrast_3d'] = (X['tbp_lv_deltaA']**2 + X['tbp_lv_deltaB']**2 + X['tbp_lv_deltaL']**2) ** 0.5
X['elongation'] = X['tbp_lv_minorAxisMM'] / (X['clin_size_long_diam_mm'] + 1e-6)
X['nevi_color_tension'] = X['tbp_lv_nevi_confidence'] * X['tbp_lv_norm_color']
X['log_area'] = X['tbp_lv_areaMM2'] ** 0.5
X['compactness'] = (X['tbp_lv_perimeterMM']**2) / (4 * 3.14159 * X['tbp_lv_areaMM2'] + 1e-6)
X['chroma_contrast'] = (X['tbp_lv_C'] - X['tbp_lv_Cext']).abs()

print(f"New engineered columns added. Total features: {X.shape[1]}")
print(X[['color_contrast_3d','elongation','nevi_color_tension','log_area','compactness','chroma_contrast']].head())

New engineered columns added. Total features: 44
   color_contrast_3d  elongation  nevi_color_tension  log_area  compactness  \
0           9.127750    0.507571            0.000000  1.775545     2.186486   
1           9.259068    0.747197            0.000000  0.958904     0.973654   
2          10.451145    0.351443            0.000000  1.806973     1.924553   
3           5.117454    0.770598           38.958815  2.465753     1.184849   
4           9.796251    0.340628            0.000000  1.449727     1.583796   

   chroma_contrast  
0         4.731520  
1         5.919770  
2         6.563120  
3         1.372177  
4         3.609240  


## Step 7 — Split Into Malignant and Benign Datasets

We reconstruct the full dataframe and split it into:
- `datasetv1.csv` — 393 confirmed malignant records (our reference profile)
- `datasetv2.csv` — all benign records (the ones we will score)

Both are saved to Google Drive.

In [8]:
df = X.copy()
df['malignant'] = y.values
df['patient_id'] = patient_id.values

df_malignant = df[df['malignant'] == 1].copy()
df_benign    = df[df['malignant'] == 0].copy()

print(f"Malignant records: {len(df_malignant)}")
print(f"Benign records: {len(df_benign)}")

df_malignant.to_csv('/content/drive/My Drive/Melanoma170/datasetv1.csv', index=False)
df_benign.to_csv('/content/drive/My Drive/Melanoma170/datasetv2.csv', index=False)

print("✅ datasetv1.csv and datasetv2.csv saved to Google Drive!")

Malignant records: 393
Benign records: 400666
✅ datasetv1.csv and datasetv2.csv saved to Google Drive!


## Step 8 — Compute the Malignant Feature Profile

We use the 393 confirmed malignant records as our reference benchmark.
For every continuous (numeric) feature, we compute:
- Mean, Median, Std, Min, Max for both benign and malignant groups

This table tells us what melanoma "looks like" in numbers.
We will use the malignant ranges in the next step to score benign records.

In [9]:
ANALYSIS_COLS = [
    'age_approx', 'clin_size_long_diam_mm', 'tbp_lv_nevi_confidence',
    'tbp_lv_norm_border', 'tbp_lv_norm_color', 'tbp_lv_area_perim_ratio',
    'tbp_lv_color_std_mean', 'tbp_lv_symm_2axis', 'tbp_lv_eccentricity',
    'tbp_lv_deltaLBnorm', 'color_contrast_3d', 'elongation',
    'nevi_color_tension', 'log_area', 'compactness', 'chroma_contrast'
]

profile = df.groupby('malignant')[ANALYSIS_COLS].agg(['mean','median','std','min','max'])

print("=== MALIGNANT PROFILE (malignant=1) ===")
print(profile.loc[1].T)
print("\n=== BENIGN PROFILE (malignant=0) ===")
print(profile.loc[0].T)

=== MALIGNANT PROFILE (malignant=1) ===
age_approx       mean      61.361323
                 median    60.000000
                 std       11.887729
                 min       20.000000
                 max       85.000000
                             ...    
chroma_contrast  mean       3.583638
                 median     3.197370
                 std        2.495542
                 min        0.041714
                 max       14.574370
Name: 1.0, Length: 80, dtype: float64

=== BENIGN PROFILE (malignant=0) ===
age_approx       mean      58.023578
                 median    60.000000
                 std       13.550804
                 min        5.000000
                 max       85.000000
                             ...    
chroma_contrast  mean       4.054594
                 median     3.960340
                 std        2.185684
                 min        0.000010
                 max       26.388339
Name: 0.0, Length: 80, dtype: float64


## Step 9 — Extract Malignant Ranges

From the malignant profile above, we extract the min and max of each feature.
These define the "malignant zone" — the range of values seen in confirmed melanoma records.

Any benign record whose feature value falls within this range gets +1 to its risk score.

In [10]:
malignant_min = df[df['malignant'] == 1][ANALYSIS_COLS].min()
malignant_max = df[df['malignant'] == 1][ANALYSIS_COLS].max()

malignant_ranges = pd.DataFrame({
    'malignant_min': malignant_min,
    'malignant_max': malignant_max
})

print("=== MALIGNANT FEATURE RANGES ===")
print(malignant_ranges)

=== MALIGNANT FEATURE RANGES ===
                         malignant_min  malignant_max
age_approx                2.000000e+01      85.000000
clin_size_long_diam_mm    1.010000e+00      18.940000
tbp_lv_nevi_confidence    1.431412e-29      99.998780
tbp_lv_norm_border        1.023068e+00      10.000000
tbp_lv_norm_color         0.000000e+00      10.000000
tbp_lv_area_perim_ratio   1.113445e+01      49.869050
tbp_lv_color_std_mean     0.000000e+00       8.009495
tbp_lv_symm_2axis         7.603306e-02       0.767033
tbp_lv_eccentricity       1.930992e-01       0.960627
tbp_lv_deltaLBnorm        3.058250e+00      30.340920
color_contrast_3d         2.727713e+00      37.714084
elongation                3.197989e-01       0.975779
nevi_color_tension        0.000000e+00     999.131858
log_area                  8.104219e-01      11.879169
compactness               8.860518e-01       3.968455
chroma_contrast           4.171414e-02      14.574370


## Step 10 (Revised) — IQR-Based Risk Scoring

Our initial risk scoring used min/max ranges from the malignant records.
This was too broad — almost every benign record fell inside the range,
resulting in 393,000+ records scoring the maximum of 16/16.

We fix this by using the IQR (Interquartile Range) instead:
- Q1 = 25th percentile of malignant records per feature
- Q3 = 75th percentile of malignant records per feature

This defines the "core zone" of confirmed malignant lesions.
A benign record only scores +1 if its value falls within Q1–Q3.
This produces a much more meaningful and spread out risk distribution.

In [11]:
# Compute Q1 and Q3 from malignant records only
malignant_q1 = df[df['malignant'] == 1][ANALYSIS_COLS].quantile(0.25)
malignant_q3 = df[df['malignant'] == 1][ANALYSIS_COLS].quantile(0.75)

malignant_iqr_zones = pd.DataFrame({
    'Q1 (25th percentile)': malignant_q1,
    'Q3 (75th percentile)': malignant_q3
})

print("=== MALIGNANT CORE ZONE (IQR) ===")
print(malignant_iqr_zones)

=== MALIGNANT CORE ZONE (IQR) ===
                         Q1 (25th percentile)  Q3 (75th percentile)
age_approx                       5.500000e+01             70.000000
clin_size_long_diam_mm           2.400000e+00              7.870000
tbp_lv_nevi_confidence           1.866102e-04             24.539140
tbp_lv_norm_border               2.404118e+00              4.924497
tbp_lv_norm_color                8.803606e-01              7.550845
tbp_lv_area_perim_ratio          1.530380e+01             24.637490
tbp_lv_color_std_mean            2.894903e-01              2.505743
tbp_lv_symm_2axis                2.090395e-01              0.400000
tbp_lv_eccentricity              6.168038e-01              0.836360
tbp_lv_deltaLBnorm               5.621653e+00             10.225560
color_contrast_3d                8.636996e+00             14.617469
elongation                       5.649670e-01              0.789570
nevi_color_tension               8.513007e-09            143.463576
log_area      

## Step 11 — Score Every Benign Record Using IQR Zone

For each benign lesion, we check how many of its feature values
fall within the malignant core zone (Q1 to Q3).

- Each matching feature adds +1 to the risk score
- Maximum possible score = 16
- Higher score = more features resemble the typical malignant lesion

In [12]:
df_benign_scored = df_benign[ANALYSIS_COLS].copy()
risk_score_iqr = pd.Series(0, index=df_benign_scored.index)

for col in ANALYSIS_COLS:
    in_range = (df_benign_scored[col] >= malignant_q1[col]) & \
               (df_benign_scored[col] <= malignant_q3[col])
    risk_score_iqr += in_range.astype(int)

df_benign = df_benign.copy()
df_benign['risk_score'] = risk_score_iqr

print(f"Risk score range: {df_benign['risk_score'].min()} to {df_benign['risk_score'].max()}")
print(f"\nRisk score distribution:")
print(df_benign['risk_score'].value_counts().sort_index())

print(f"\n=== RISK TIERS ===")
print(f"High risk   (score >= 12): {(df_benign['risk_score'] >= 12).sum():,} records")
print(f"Medium risk  (score 8-11): {((df_benign['risk_score'] >= 8) & (df_benign['risk_score'] < 12)).sum():,} records")
print(f"Low risk      (score < 8): {(df_benign['risk_score'] < 8).sum():,} records")

Risk score range: 0 to 16

Risk score distribution:
risk_score
0        18
1       145
2       959
3      2849
4      7454
5     13257
6     22418
7     32510
8     39888
9     45061
10    47364
11    49238
12    49596
13    43264
14    29760
15    13297
16     3588
Name: count, dtype: int64

=== RISK TIERS ===
High risk   (score >= 12): 139,505 records
Medium risk  (score 8-11): 181,551 records
Low risk      (score < 8): 79,610 records


## Step 12 — Rank Benign Records and Save Final Dataset

We sort all benign records from highest to lowest risk score.
The top of this list = benign lesions whose features most closely
resemble the core profile of confirmed malignant melanoma.

We then save the final combined dataset as datasetv3.csv.

In [13]:
# Rank benign records
df_benign_ranked = df_benign.sort_values('risk_score', ascending=False)

print("=== TOP 20 HIGHEST RISK BENIGN LESIONS ===")
print(df_benign_ranked[['risk_score'] + ANALYSIS_COLS].head(20))

# Save final dataset
df_malignant['risk_score'] = None
df_final = pd.concat([df_benign, df_malignant], ignore_index=True)
df_final.to_csv('/content/drive/My Drive/Melanoma170/datasetv3.csv', index=False)

print(f"\n✅ datasetv3.csv saved!")
print(f"Total records: {len(df_final):,}")
print(f"Benign with risk scores: {len(df_benign):,}")
print(f"Malignant (reference): {len(df_malignant):,}")

=== TOP 20 HIGHEST RISK BENIGN LESIONS ===
        risk_score  age_approx  clin_size_long_diam_mm  \
74791           16        60.0                    2.70   
141570          16        55.0                    3.00   
197967          16        55.0                    2.71   
351924          16        60.0                    4.54   
301436          16        65.0                    3.05   
301467          16        65.0                    4.61   
141373          16        65.0                    3.38   
80220           16        55.0                    3.15   
41689           16        65.0                    7.00   
198084          16        70.0                    4.46   
28474           16        55.0                    2.93   
198108          16        65.0                    3.38   
301701          16        65.0                    2.82   
301705          16        60.0                    3.98   
141053          16        65.0                    3.38   
256463          16        70.